# Guide Evaluation Analysis

Analyzes CSV output from `guide-eval` to evaluate distance metrics as predictors for equality saturation reachability.

Run with: `uv run jupyter notebook guide_eval.ipynb`

In [ ]:
from pathlib import Path

import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.stats import spearmanr, kendalltau
from sklearn.linear_model import LinearRegression

plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 8)})

In [ ]:
# ── Load data ──────────────────────────────────────────────────────────
# Point this at the runs/ directory produced by guide-eval
RUNS_DIR = Path("../runs")

# Auto-detect: pick the most recent CSV matching the run-* pattern
candidates = sorted(RUNS_DIR.glob("run-*.csv"), key=lambda p: p.stat().st_mtime)
if not candidates:
    raise FileNotFoundError(f"No run-*.csv found in {RUNS_DIR.resolve()}")
csv_file = candidates[-1]
print(f"Using: {csv_file}")

df = pl.read_csv(csv_file)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# ── Overview ───────────────────────────────────────────────────────────
df.schema

In [ ]:
# ── Predictor columns ─────────────────────────────────────────────────
# All numeric columns that could predict iteration count.
# Extend this list if you add new distance metrics.
PREDICTOR_CANDIDATES = [
    "zs_rank",
    "structural_rank",
    "zs_distance",
    "structural_overlap",
    "structural_zs_sum",
]
TARGET = "iterations_to_reach"

# Keep only columns that actually exist in this CSV
predictors = [c for c in PREDICTOR_CANDIDATES if c in df.columns]
print(f"Predictors found: {predictors}")

## Per-goal analysis

In [ ]:
goals = df["goal"].unique().to_list()
print(f"{len(goals)} goal(s) in dataset")

In [ ]:
# ── Reachability summary per goal ──────────────────────────────────────
summary_rows = []
for goal in goals:
    g = df.filter(pl.col("goal") == goal)
    total = len(g)
    reached = g.filter(pl.col("reached"))
    n_reached = len(reached)
    pct = 100.0 * n_reached / max(total, 1)
    iters = reached[TARGET].drop_nulls()
    summary_rows.append(
        {
            "goal": goal[:80] + ("..." if len(goal) > 80 else ""),
            "total": total,
            "reached": n_reached,
            "reached_%": round(pct, 1),
            "iter_min": iters.min() if len(iters) > 0 else None,
            "iter_median": iters.median() if len(iters) > 0 else None,
            "iter_max": iters.max() if len(iters) > 0 else None,
        }
    )

summary = pl.DataFrame(summary_rows)
summary

## Correlation analysis

For each predictor, compute Spearman and Kendall-tau correlation with `iterations_to_reach` (reached guides only).

In [ ]:
def compute_correlations(
    data: pl.DataFrame, predictors: list[str], target: str
) -> pl.DataFrame:
    """Compute Spearman rho, Kendall tau, and OLS R² for each predictor vs target."""
    reached = data.filter(pl.col("reached")).drop_nulls(subset=[target])
    rows = []
    for pred in predictors:
        if pred not in reached.columns:
            continue
        col = reached[pred].drop_nulls()
        tgt = reached[target].drop_nulls()
        # Align lengths
        n = min(len(col), len(tgt))
        if n < 3:
            continue
        x = col[:n].to_numpy().astype(float)
        y = tgt[:n].to_numpy().astype(float)

        rho, rho_p = spearmanr(x, y)
        tau, tau_p = kendalltau(x, y)

        # OLS
        reg = LinearRegression().fit(x.reshape(-1, 1), y)
        r2 = reg.score(x.reshape(-1, 1), y)
        slope = reg.coef_[0]
        intercept = reg.intercept_

        # Standard error of slope
        y_pred = reg.predict(x.reshape(-1, 1))
        residuals = y - y_pred
        mse = np.sum(residuals**2) / max(n - 2, 1)
        ss_xx = np.sum((x - x.mean()) ** 2)
        se_slope = np.sqrt(mse / ss_xx) if ss_xx > 1e-12 else 0.0

        rows.append(
            {
                "predictor": pred,
                "n": n,
                "spearman_rho": round(rho, 4),  # pyright: ignore[reportCallIssue, reportArgumentType]
                "spearman_p": round(rho_p, 6),  # pyright: ignore[reportCallIssue, reportArgumentType]
                "kendall_tau": round(tau, 4),  # pyright: ignore[reportCallIssue, reportArgumentType]
                "kendall_p": round(tau_p, 6),  # pyright: ignore[reportCallIssue, reportArgumentType]
                "R²": round(r2, 4),
                "slope": round(slope, 4),
                "SE_slope": round(se_slope, 4),
                "intercept": round(intercept, 4),  # pyright: ignore[reportCallIssue, reportArgumentType]
            }
        )
    return pl.DataFrame(rows)

In [ ]:
# ── Per-goal correlations ──────────────────────────────────────────────
from IPython.display import display, Markdown

for goal in goals:
    g = df.filter(pl.col("goal") == goal)
    n_reached = len(g.filter(pl.col("reached")))
    display(Markdown(f"### Goal (n_reached={n_reached}): `{goal[:100]}`"))
    corr = compute_correlations(g, predictors, TARGET)
    if len(corr) > 0:
        display(corr)
    else:
        display(Markdown("*Not enough reached guides for correlation.*"))

## Scatter plots: predictor vs iterations to reach

In [ ]:
def plot_predictor_vs_iters(
    data: pl.DataFrame,
    predictor: str,
    target: str = TARGET,
    title_suffix: str = "",
    ax: plt.Axes | None = None,  # pyright: ignore[reportPrivateImportUsage]
):
    """Scatter plot of predictor vs iterations, with reached/timed-out markers and OLS line."""
    if ax is None:
        _, ax = plt.subplots()

    reached = data.filter(pl.col("reached")).drop_nulls(subset=[target])
    timed_out = data.filter(~pl.col("reached"))

    # Reached points
    x_r = reached[predictor].to_numpy().astype(float)
    y_r = reached[target].to_numpy().astype(float)
    ax.scatter(
        x_r, y_r, c="steelblue", s=20, alpha=0.6, label=f"reached (n={len(x_r)})"
    )

    # Timed-out points (plotted at max_iters + 1)
    if len(timed_out) > 0:
        x_t = timed_out[predictor].to_numpy().astype(float)
        timeout_y = y_r.max() + 1 if len(y_r) > 0 else 1
        ax.scatter(
            x_t,
            np.full_like(x_t, timeout_y),
            c="red",
            marker="x",
            s=20,
            alpha=0.5,
            label=f"timed out (n={len(x_t)})",
        )
        ax.axhline(timeout_y, color="red", ls="--", alpha=0.3, lw=1)

    # OLS regression line (reached only)
    if len(x_r) >= 2:
        reg = LinearRegression().fit(x_r.reshape(-1, 1), y_r)
        x_line = np.linspace(x_r.min(), x_r.max(), 100)
        ax.plot(
            x_line,
            reg.predict(x_line.reshape(-1, 1)),
            c="green",
            lw=2,
            alpha=0.7,
            label=f"OLS (R²={reg.score(x_r.reshape(-1, 1), y_r):.3f})",
        )

    ax.set_xlabel(predictor)
    ax.set_ylabel(target)
    ax.set_title(f"{predictor} vs {target}{title_suffix}")
    ax.legend(fontsize=8)
    return ax

In [ ]:
# ── All predictors, all goals pooled ───────────────────────────────────
n_pred = len(predictors)
cols = min(n_pred, 3)
rows = (n_pred + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows), squeeze=False)

for i, pred in enumerate(predictors):
    ax = axes[i // cols][i % cols]
    plot_predictor_vs_iters(df, pred, ax=ax, title_suffix=" (all goals)")

# Hide unused axes
for j in range(n_pred, rows * cols):
    axes[j // cols][j % cols].set_visible(False)

fig.tight_layout()
plt.show()

In [ ]:
# ── Per-goal scatter plots ─────────────────────────────────────────────
for goal in goals:
    g = df.filter(pl.col("goal") == goal)
    n_reached = len(g.filter(pl.col("reached")))
    if n_reached < 3:
        print(f"Skipping goal (only {n_reached} reached): {goal[:80]}")
        continue

    n_pred = len(predictors)
    cols = min(n_pred, 3)
    rows = (n_pred + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows), squeeze=False)
    for i, pred in enumerate(predictors):
        plot_predictor_vs_iters(
            g, pred, ax=axes[i // cols][i % cols], title_suffix=f"\ngoal: {goal[:60]}"
        )
    for j in range(n_pred, rows * cols):
        axes[j // cols][j % cols].set_visible(False)
    fig.tight_layout()
    plt.show()

In [ ]:
# ── Box plots: predictor values for reached vs not reached ──────────
n_pred = len(predictors)
cols = min(n_pred, 3)
rows = (n_pred + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows), squeeze=False)

for i, pred in enumerate(predictors):
    ax = axes[i // cols][i % cols]
    reached_vals = df.filter(pl.col("reached"))[pred].drop_nulls().to_numpy()
    unreached_vals = df.filter(~pl.col("reached"))[pred].drop_nulls().to_numpy()
    ax.boxplot(
        [reached_vals, unreached_vals],
        tick_labels=["reached", "not reached"],
        patch_artist=True,
        boxprops=dict(facecolor="lightblue"),
    )
    ax.set_title(pred)
    ax.set_ylabel(pred)

for j in range(n_pred, rows * cols):
    axes[j // cols][j % cols].set_visible(False)

fig.suptitle("Predictor distributions: reached vs not reached", fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
# ── Box plots: predictor values for reached vs not reached ──────────
fig, axes = plt.subplots(
    1, len(predictors), figsize=(5 * len(predictors), 5), squeeze=False
)

for i, pred in enumerate(predictors):
    ax = axes[0][i]
    reached_vals = df.filter(pl.col("reached"))[pred].drop_nulls().to_numpy()
    unreached_vals = df.filter(~pl.col("reached"))[pred].drop_nulls().to_numpy()
    ax.boxplot(
        [reached_vals, unreached_vals],
        tick_labels=["reached", "not reached"],
        patch_artist=True,
        boxprops=dict(facecolor="lightblue"),
    )
    ax.set_title(pred)
    ax.set_ylabel(pred)

fig.suptitle("Predictor distributions: reached vs not reached", fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
# ── Histograms of iterations_to_reach (reached only) ──────────────────
reached_all = df.filter(pl.col("reached"))
iters_vals = reached_all[TARGET].drop_nulls().to_numpy()

if len(iters_vals) > 0:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(
        iters_vals,
        bins=max(1, int(iters_vals.max() - iters_vals.min() + 1)),
        color="steelblue",
        edgecolor="white",
    )
    ax.set_xlabel("iterations_to_reach")
    ax.set_ylabel("count")
    ax.set_title("Distribution of iterations to reach goal")
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    fig.tight_layout()
    plt.show()
else:
    print("No reached guides to plot.")

## Iteration breakdown table

In [ ]:
# ── Breakdown by iterations to reach ──────────────────────────────────
# Group by iteration count (None for unreached)
breakdown_rows = []
for goal in goals:
    g = df.filter(pl.col("goal") == goal)
    goal_short = goal[:60] + ("..." if len(goal) > 60 else "")

    # Group reached by iteration count
    reached = g.filter(pl.col("reached"))
    if len(reached) > 0:
        by_iter = reached.group_by(TARGET).agg(pl.len().alias("count")).sort(TARGET)
        for row in by_iter.iter_rows(named=True):
            breakdown_rows.append(
                {
                    "goal": goal_short,
                    "iterations": row[TARGET],
                    "count": row["count"],
                }
            )

    # Unreached
    n_unreached = len(g.filter(~pl.col("reached")))
    if n_unreached > 0:
        breakdown_rows.append(
            {
                "goal": goal_short,
                "iterations": None,
                "count": n_unreached,
            }
        )

if breakdown_rows:
    pl.DataFrame(breakdown_rows)
else:
    print("No data.")

## Predictor comparison heatmap

In [ ]:
# ── Spearman correlation heatmap across all predictors ─────────────────
reached = df.filter(pl.col("reached")).drop_nulls(subset=[TARGET])
cols_for_heatmap = [p for p in predictors if p in reached.columns] + [TARGET]
mat = reached.select(cols_for_heatmap).to_numpy().astype(float)

n_cols = len(cols_for_heatmap)
corr_matrix = np.zeros((n_cols, n_cols))
for i in range(n_cols):
    for j in range(n_cols):
        corr_matrix[i, j], _ = spearmanr(mat[:, i], mat[:, j])  # pyright: ignore[reportArgumentType, reportCallIssue]

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr_matrix, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(n_cols))
ax.set_yticks(range(n_cols))
ax.set_xticklabels(cols_for_heatmap, rotation=45, ha="right")
ax.set_yticklabels(cols_for_heatmap)

# Annotate cells
for i in range(n_cols):
    for j in range(n_cols):
        ax.text(
            j,
            i,
            f"{corr_matrix[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=9,
            color="white" if abs(corr_matrix[i, j]) > 0.6 else "black",
        )

fig.colorbar(im, ax=ax, label="Spearman ρ")
ax.set_title("Spearman correlation matrix (reached guides)")
fig.tight_layout()
plt.show()